# DCSM-Simplified: wave 2 (human labels)

Wave 1 learned from YOLO pseudo-labels (teacher 69 %, best student ~0.59 on board-like images).
Wave 2 adds the **2,056 human-labelled DC-Ops images (32.5k boxes)**:

1. **teacher_v2**: crops from human labels + wave-1 crops for the missing classes + background; two ConvNeXt-Base fold models give *out-of-fold* soft labels; scored on the **human-labelled validation split**.
2. **Students** (5 runs): MobileNetV2 for the Ethos-U85 with distillation, board-camera augmentation, mixup and weight EMA; ablations included.
3. **Keyword spotters** (2 runs): retrained with the board's auto-gain emulated (the E8 audio HAL boosts each window toward full scale), int8 calibrated on 512 clips.

Same setup as wave 1: GPU runtime, `HF_TOKEN` secret, keep the tab open. Re-running skips finished jobs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/dcops_e8/wave2'
except Exception as e:
    print('Drive not mounted, using local disk:', e)
    OUT = '/content/dcops_e8/wave2'
os.makedirs(f'{OUT}/logs', exist_ok=True)
LOCAL = '/content/v2'   # multi-GB crops stay on the fast local disk
print('outputs ->', OUT)

In [ ]:
!pip -q install timm transformers soundfile huggingface_hub
from huggingface_hub import hf_hub_download
import shutil
os.makedirs('/content/jobs', exist_ok=True)
for f in ['teacher_v2.py', 'student.py', 'kws.py']:
    shutil.copy(hf_hub_download('abhijitbetigeri/dcops-e8-jobs', f, force_download=True), f'/content/jobs/{f}')
print(os.listdir('/content/jobs'))

In [ ]:
import subprocess, time, torch, glob, json
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
SLOTS = 6 if vram > 60 else 4 if vram > 35 else 3 if vram > 20 else 2
print(f'GPU {torch.cuda.get_device_name(0)}, {vram:.0f} GB -> {SLOTS} jobs at a time')

CROPS, META = f'{LOCAL}/crops_v2.npz', f'{LOCAL}/meta_v2.json'
S = f'--crops {CROPS} --meta {META} --res 160'
JOBS = [  # name, command, done-marker, needs teacher crops
  ('teacher_v2',   f'teacher_v2.py --out {LOCAL}',                                    f'{OUT}/teacher_v2/meta_v2.json', False),
  ('kws_m_agc',    'kws.py --name dscnn_m_agc --ch 128 --kd-alpha 0.5 --agc 0.8 --calib-n 512 --epochs 60 --data /content/sc_m', f'{OUT}/kws_dscnn_m_agc/metrics.json', False),
  ('kws_s_agc',    'kws.py --name dscnn_s_agc --ch 64 --kd-alpha 0.5 --agc 0.8 --calib-n 512 --epochs 60 --data /content/sc_s',  f'{OUT}/kws_dscnn_s_agc/metrics.json', False),
  ('v2_kd_mix',    f'student.py --name v2_r160_kd_mix {S} --kd-alpha 0.7 --mixup 0.5 --ema 0.999 --epochs 40 --batch 128', f'{OUT}/run_v2_r160_kd_mix/metrics.json', True),
  ('v2_kd',        f'student.py --name v2_r160_kd {S} --kd-alpha 0.7 --ema 0.999 --epochs 40 --batch 128',                f'{OUT}/run_v2_r160_kd/metrics.json', True),
  ('v2_nokd',      f'student.py --name v2_r160_nokd {S} --kd-alpha 0 --ema 0.999 --epochs 40 --batch 128',                f'{OUT}/run_v2_r160_nokd/metrics.json', True),
  ('v2_noaug',     f'student.py --name v2_r160_noaug {S} --kd-alpha 0.7 --mixup 0.5 --ema 0.999 --domain-aug 0 --epochs 40 --batch 128', f'{OUT}/run_v2_r160_noaug/metrics.json', True),
  ('v2_kd_mix_e80',f'student.py --name v2_r160_kd_mix_e80 {S} --kd-alpha 0.7 --mixup 0.5 --ema 0.9995 --epochs 80 --batch 128', f'{OUT}/run_v2_r160_kd_mix_e80/metrics.json', True),
]
RUN = {'v2_kd_mix': 'v2_r160_kd_mix', 'v2_kd': 'v2_r160_kd', 'v2_nokd': 'v2_r160_nokd', 'v2_noaug': 'v2_r160_noaug',
       'v2_kd_mix_e80': 'v2_r160_kd_mix_e80', 'kws_m_agc': 'dscnn_m_agc', 'kws_s_agc': 'dscnn_s_agc'}

def after(name):
    if name == 'teacher_v2':
        os.makedirs(f'{OUT}/teacher_v2', exist_ok=True)
        shutil.copy(META, f'{OUT}/teacher_v2/meta_v2.json')   # crops stay local (GBs)
    elif name.startswith('kws'):
        shutil.copytree(f'/tmp/kws_{RUN[name]}', f'{OUT}/kws_{RUN[name]}', dirs_exist_ok=True)
    else:
        shutil.copytree(f'/tmp/run_{RUN[name]}', f'{OUT}/run_{RUN[name]}', dirs_exist_ok=True)

# the crops live on local disk: if the runtime was reset, rebuild them (rerun the teacher)
if not os.path.exists(CROPS) and os.path.exists(f'{OUT}/teacher_v2/meta_v2.json'):
    os.remove(f'{OUT}/teacher_v2/meta_v2.json')
pending = [j for j in JOBS if not os.path.exists(j[2])]
print('to run:', [j[0] for j in pending])
running, t0 = {}, time.time()
while pending or running:
    teacher_done = os.path.exists(CROPS) and os.path.exists(META)
    for j in list(pending):
        if len(running) >= SLOTS: break
        if j[3] and not teacher_done: continue
        log = open(f"{OUT}/logs/{j[0]}.log", 'a')
        running[j[0]] = (subprocess.Popen(f'python /content/jobs/{j[1]}', shell=True, stdout=log, stderr=subprocess.STDOUT), j)
        pending.remove(j); print(time.strftime('%H:%M'), 'started', j[0])
    for name, (p, j) in list(running.items()):
        if p.poll() is not None:
            del running[name]
            if p.returncode == 0:
                try: after(name); print(time.strftime('%H:%M'), 'done', name)
                except Exception as e: print('copy failed', name, e)
            else:
                print(time.strftime('%H:%M'), f'FAILED {name} (exit {p.returncode}); tail of log:')
                print(open(f'{OUT}/logs/{name}.log').read()[-1500:])
    time.sleep(60)
    if int(time.time() - t0) % 1800 < 60:
        print(time.strftime('%H:%M'), 'running:', list(running), 'pending:', [j[0] for j in pending])
print('ALL DONE in', round((time.time() - t0) / 3600, 1), 'h')

In [ ]:
for f in sorted(glob.glob(f'{OUT}/run_*/metrics.json')):
    m = json.load(open(f)); print(f"{m['name']:22s} {m.get('metric','')}: {m['val_acc']:.3f}  board-sim {m['val_acc_board_sim']:.3f}")
for f in sorted(glob.glob(f'{OUT}/kws_*/metrics.json')):
    m = json.load(open(f)); print(f"{m['name']:22s} test {m['test_acc']:.4f}  with board auto-gain {m.get('test_acc_board_agc', float('nan')):.4f}")